## AuxTel Mount fails - 03-Mar-21

In this notebook, investigate closed dome mount tracking on 02-Mar-21

In [1]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.time import Time, TimeDelta
from astropy.coordinates import AltAz, ICRS, EarthLocation, Angle, FK5
import astropy.units as u
from lsst_efd_client import EfdClient

In [2]:
# Set Cerro Pachon location
location = EarthLocation.from_geodetic(lon=-70.747698*u.deg,
                                       lat=-30.244728*u.deg,
                                       height=2663.0*u.m)

In [3]:
# Get EFD client and bring in Lupton's unpacking code
client = EfdClient('summit_efd')

def merge_packed_time_series(packed_dataframe, base_field, stride=1, 
                             ref_timestamp_col="cRIO_timestamp", internal_time_scale="tai"):
    """Select fields that are time samples and unpack them into a dataframe.
            Parameters
            ----------
            packedDF : `pandas.DataFrame`
                packed data frame containing the desired data
            base_field :  `str`
                Base field name that will be expanded to query all
                vector entries.
            stride : `int`, optional
                Only use every stride value when unpacking.  Must be a factor
                of the number of packed values.
                (1 by default)
            ref_timestamp_col : `str`, optional
                Name of the field name to use to assign timestamps to unpacked
                vector fields (default is 'cRIO_timestamp').
            internal_time_scale : `str`, optional
                Time scale to use when converting times to internal formats
                ('tai' by default). Equivalent to EfdClient.internal_scale
        Returns
            -------
            result : `pandas.DataFrame`
                A `pandas.DataFrame` containing the results of the query.
            """
    
    packed_fields = [k for k in packed_dataframe.keys() if k.startswith(base_field)]
    packed_fields = sorted(packed_fields, key=lambda k: int(k[len(base_field):]))  # sort by pack ID
    npack = len(packed_fields)
    if npack%stride != 0:
        raise RuntimeError(f"Stride must be a factor of the number of packed fields: {stride} v. {npack}")
    packed_len = len(packed_dataframe)
    n_used = npack//stride   # number of raw fields being used
    output = np.empty(n_used*packed_len)
    times = np.empty_like(output, dtype=packed_dataframe[ref_timestamp_col][0])
    
    if packed_len == 1:
        dt = 0
    else:
        dt = (packed_dataframe[ref_timestamp_col][1] - packed_dataframe[ref_timestamp_col][0])/npack
    for i in range(0, npack, stride):
        i0 = i//stride
        output[i0::n_used] = packed_dataframe[f"{base_field}{i}"]
        times[i0::n_used] = packed_dataframe[ref_timestamp_col] + i*dt
     
    timestamps = Time(times, format='unix', scale=internal_time_scale).datetime64
    return pd.DataFrame({base_field:output, "times":times}, index=timestamps)

In [4]:
# These are for finding the timestamps of the scripts
start = Time('2021-04-09T16:46:00') #this is UTC
end = Time('2021-04-09T16:49:00')

In [ ]:
# Script 200053 started tracking for 60 seconds
timestamp = f"time >= '{start}+00:00' AND time <= '{end}+00:00'"
script_id = 200053
track_time = 60.0
query = f'SELECT "heartbeat", "ScriptID" FROM "efd"."autogen"."lsst.sal.Script.logevent_heartbeat"\
    WHERE {timestamp} AND ScriptID = {script_id}'

script_time_frame = await client.influx_client.query(query)


In [ ]:
time_start = f"{script_time_frame.index[0]}".replace(" ", "T")
time_end = f"{script_time_frame.index[-1]}".replace(" ", "T")
timestamp = f"time >= '{time_start}' AND time <= '{time_end}'"
query = f'SELECT "inPosition" FROM "efd"."autogen"."lsst.sal.ATMCS.logevent_allAxesInPosition"\
    WHERE {timestamp} and inPosition = true'

tracking_started = await client.influx_client.query(query)

In [ ]:
time_tracking_started = Time(tracking_started.index[-1])
print(tracking_started, time_tracking_started.tai)

In [ ]:
# Get the data
for time_shift in [0.0, -37.0]:

    ts = time_tracking_started.tai + TimeDelta(time_shift, format='sec')
    te = ts + TimeDelta(track_time, format='sec')
    print(ts.isot, te.isot)

    mount_position = await client.select_time_series("lsst.sal.ATMCS.mount_AzEl_Encoders", ['*'],
                                              ts, te)
    nasmyth_position = await client.select_time_series("lsst.sal.ATMCS.mount_Nasmyth_Encoders", ['*'],
                                              ts, te)

    az = merge_packed_time_series(mount_position, 'azimuthCalculatedAngle', stride=1)
    el = merge_packed_time_series(mount_position, 'elevationCalculatedAngle', stride=1)
    rot = merge_packed_time_series(nasmyth_position, 'nasmyth2CalculatedAngle', stride=1) 

    # Plot it
    fig = plt.figure(figsize = (16,6))
    plt.suptitle(f"Mount Tracking - Script {script_id}", fontsize = 18)
    # Azimuth axis
    plt.subplot(1,3,1)
    ax1 = az['azimuthCalculatedAngle'].plot(legend=True, color='red')
    ax1.set_title("Azimuth axis", fontsize=16)
    ax1.axvline(ts.isot, color="green", linestyle="--")
    ax1.axvline(te.isot, color="red", linestyle="--")
    ax1.set_ylabel("Degrees")

    # Elevation axis
    plt.subplot(1,3,2)
    ax2 = el['elevationCalculatedAngle'].plot(legend=True, color='green')
    ax2.set_title("Elevation axis", fontsize=16)
    ax2.axvline(ts.isot, color="green", linestyle="--")
    ax2.axvline(te.isot, color="red", linestyle="--")

    # Nasmyth2 rotator axis
    plt.subplot(1,3,3)
    ax3 = rot['nasmyth2CalculatedAngle'].plot(legend=True, color='blue')
    ax3.set_title("Nasmyth2 axis", fontsize=16)
    ax3.axvline(ts.isot, color="green", linestyle="--")
    ax3.axvline(te.isot, color="red", linestyle="--")

    plt.savefig(f"/home/craiglagegit/DATA/Tracking_Timebase_{script_id}_Shift_{time_shift}_20Feb21.pdf")
